In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification : une Qiskit Function par Multiverse Computing
> **Note:** * Les Qiskit Functions sont une fonctionnalité expérimentale disponible uniquement pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles d'évoluer.

## Vue d'ensemble
Avec la fonction « Singularity Machine Learning - Classification », tu peux résoudre des problèmes réels d'apprentissage automatique sur du matériel quantique sans avoir besoin d'expertise quantique. Cette fonction applicative, basée sur des méthodes ensemblistes, est un classificateur hybride. Elle exploite des méthodes classiques comme le boosting, le bagging et le stacking pour l'entraînement initial de l'ensemble. Ensuite, des algorithmes quantiques tels que le solveur variationnel quantique des valeurs propres (VQE) et l'algorithme d'optimisation approximative quantique (QAOA) sont utilisés pour améliorer la diversité, les capacités de généralisation et la complexité globale de l'ensemble entraîné.

Contrairement à d'autres solutions de machine learning quantique, cette fonction est capable de gérer des jeux de données à grande échelle contenant des millions d'exemples et de caractéristiques, sans être limitée par le nombre de qubits du QPU cible. Le nombre de qubits détermine uniquement la taille de l'ensemble qui peut être entraîné. Elle est également très flexible et peut être utilisée pour résoudre des problèmes de classification dans un large éventail de domaines, notamment la finance, la santé et la cybersécurité.
Elle atteint systématiquement de hautes précisions sur des problèmes classiquement difficiles impliquant des jeux de données de grande dimension, bruités et déséquilibrés.
![Comment ça fonctionne](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Elle est conçue pour :
1. Les ingénieurs et les data scientists d'entreprises souhaitant enrichir leurs offres technologiques en intégrant le machine learning quantique dans leurs produits et services,
2. Les chercheurs dans des laboratoires de recherche quantique explorant les applications du machine learning quantique et cherchant à tirer parti de l'informatique quantique pour des tâches de classification, et
3. Les étudiants et les enseignants dans des établissements d'enseignement, dans des cours tels que le machine learning, qui souhaitent démontrer les avantages de l'informatique quantique.

L'exemple suivant illustre ses différentes fonctionnalités, notamment `create`, `list`, `fit` et `predict`, et en démontre l'utilisation sur un problème synthétique composé de deux demi-cercles entrelacés, un problème notoirement difficile en raison de sa frontière de décision non linéaire.


## Description de la fonction
Cette Qiskit Function permet aux utilisateurs de résoudre des problèmes de classification binaire à l'aide du classificateur ensembliste amélioré quantiquement de Singularity. En coulisse, elle utilise une approche hybride pour entraîner classiquement un ensemble de classificateurs sur le jeu de données étiqueté, puis l'optimise pour une diversité et une généralisation maximales en utilisant l'algorithme d'optimisation approximative quantique (QAOA) sur les QPU IBM&reg;. Grâce à une interface conviviale, les utilisateurs peuvent configurer un classificateur selon leurs besoins, l'entraîner sur le jeu de données de leur choix et l'utiliser pour faire des prédictions sur un jeu de données inédit.

Pour résoudre un problème de classification générique :
1. Prétraite le jeu de données et divise-le en ensembles d'entraînement et de test. Optionnellement, tu peux subdiviser l'ensemble d'entraînement en ensembles d'entraînement et de validation. Cela peut être réalisé à l'aide de [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Si l'ensemble d'entraînement est déséquilibré, tu peux le rééchantillonner pour équilibrer les classes à l'aide de [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Téléverse les ensembles d'entraînement, de validation et de test séparément dans le stockage de la fonction à l'aide de la méthode `file_upload` du catalogue, en lui passant le chemin pertinent à chaque fois.
4. Initialise le classificateur quantique en utilisant l'action `create` de la fonction, qui accepte des hyperparamètres tels que le nombre et les types d'apprenants, la régularisation (valeur lambda) et les options d'optimisation, notamment le nombre de couches, le type d'optimiseur classique, le backend quantique, etc.
5. Entraîne le classificateur quantique sur l'ensemble d'entraînement en utilisant l'action `fit` de la fonction, en lui passant l'ensemble d'entraînement étiqueté et l'ensemble de validation le cas échéant.
6. Effectue des prédictions sur le jeu de test inédit à l'aide de l'action `predict` de la fonction.
## Approche par actions
La fonction utilise une approche par actions. Tu peux la concevoir comme un environnement virtuel dans lequel tu utilises des actions pour effectuer des tâches ou modifier son état. Actuellement, elle propose les actions suivantes : [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) et [create_fit_predict](#7-create-fit-predict). L'exemple suivant illustre l'action `create_fit_predict`.

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

L'action `list` récupère tous les classificateurs stockés au format `*.pkl.tar` depuis le répertoire de données partagé. Tu peux également accéder au contenu de ce répertoire en utilisant la méthode `catalog.files()`. En général, l'action list recherche les fichiers avec l'extension `*.pkl.tar` dans le répertoire de données partagé et les renvoie sous forme de liste.

#### Entrées

|     Nom     |    Type     | Description | Requis |
|-------------|-------------|-------------|--------|
| `action` | `str` | Le nom de l'action parmi `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` et `delete`. | Oui |

#### Utilisation

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
L'action `create` crée un classificateur du type `quantum_classifier` spécifié en utilisant les paramètres fournis, et le sauvegarde dans le répertoire de données partagé.

> **Note:** La fonction ne supporte actuellement que le `QuantumEnhancedEnsembleClassifier`.
#### Entrées
|     Nom     |    Type     | Description | Requis | Défaut |
|-------------|-------------|-------------|--------|--------|
| `action` | `str` | Le nom de l'action parmi `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` et `delete`. | Oui | - |
| `name` | `str` | Le nom du classificateur quantique, par ex. `spam_classifier`. | Oui | - |
| `instance` | `str` | Instance IBM. | Oui | - |
| `backend_name` | `str` | Ressource de calcul IBM. Par défaut `None`, ce qui signifie que le backend avec le moins de tâches en attente sera utilisé. | Non | `None` |
| `quantum_classifier` | `str` | Le type du classificateur quantique, c'est-à-dire `QuantumEnhancedEnsembleClassifier`. | Non | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Le nombre d'apprenants dans l'ensemble. | Non | `10` |
| `learners_types` | `list` | Types d'apprenants. Parmi les types supportés : `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` et `LogisticRegression`. Plus de détails sur chacun sont disponibles dans la [documentation scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Non | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proportions de chaque type d'apprenant dans l'ensemble. | Non | `[1.0]` |
| `learners_options` | `list` | Options pour chaque type d'apprenant dans l'ensemble. Pour une liste complète des options correspondant aux types d'apprenants choisis, consulte la [documentation scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Non | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` ou `list` | Type(s) de régularisation à utiliser : `onsite` ou `alpha`. `onsite` contrôle le terme onsite où des valeurs plus élevées conduisent à des ensembles plus creux. `alpha` contrôle le compromis entre les termes d'interaction et onsite où des valeurs plus basses conduisent à des ensembles plus creux. Si une liste est fournie, des modèles seront entraînés pour chaque type et le meilleur sera sélectionné. | Non | `onsite` |
| `regularization` | `str` ou `float` ou `list` | Valeur de régularisation. Comprise entre `0` et `+inf` si `regularization_type` est `onsite`. Comprise entre `0` et `1` si `regularization_type` est `alpha`. Si défini à `auto`, la régularisation automatique est utilisée — le paramètre de régularisation optimal est trouvé par recherche binaire avec le ratio désiré de classificateurs sélectionnés par rapport au total (`regularization_desired_ratio`) et la borne supérieure du paramètre de régularisation (`regularization_upper_bound`). Si une liste est fournie, des modèles seront entraînés pour chaque valeur et le meilleur sera sélectionné. | Non | `0.01` |
| `regularization_desired_ratio` | `float` ou `list` | Ratio(s) désiré(s) de classificateurs sélectionnés par rapport au total pour la régularisation automatique. Si une liste est fournie, des modèles seront entraînés pour chaque ratio et le meilleur sera sélectionné. | Non | `0.75` |
| `regularization_upper_bound` | `float` ou `list` | Borne(s) supérieure(s) du paramètre de régularisation lors de l'utilisation de la régularisation automatique. Si une liste est fournie, des modèles seront entraînés pour chaque borne supérieure et le meilleur sera sélectionné. | Non | `200` |
| `weight_update_method` | `str` | Méthode de mise à jour des poids des échantillons parmi `logarithmic` et `quadratic`. | Non | `logarithmic` |
| `sample_scaling` | `boolean` | Indique si la mise à l'échelle des échantillons doit être appliquée. | Non | `False` |
| `prediction_scaling` | `float` | Facteur de mise à l'échelle pour les prédictions. | Non | `None` |
| `optimizer_options` | `dictionary` | Options de l'optimiseur QAOA. Une liste des options disponibles est présentée plus loin dans cette documentation. | Non | ... |
| `voting` | `str` | Utilise le vote majoritaire (`hard`) ou la moyenne des probabilités (`soft`) pour agréger les prédictions/probabilités des apprenants. | Non | `hard` |
| `prob_threshold` | `float` | Seuil de probabilité optimal. | Non | `0.5` |
| `random_state` | `integer` | Contrôle l'aléatoire pour la reproductibilité. | Non | `None` |


- De plus, les `optimizer_options` sont listées comme suit :

|     Nom     |    Type     | Description | Requis | Défaut |
|-------------|-------------|-------------|--------|--------|
| `num_solutions` | `integer` | Le nombre de solutions | Non | `1024` |
| `reps` | `integer` | Le nombre de répétitions | Non | `4` |
| `sparsify` | `float` | Le seuil de sparsification | Non | `0.001` |
| `theta` | `float` | La valeur initiale de theta, un paramètre variationnel du QAOA | Non | `None` |
| `simulator` | `boolean` | Indique s'il faut utiliser un simulateur ou un QPU | Non | `False` |
| `classical_optimizer` | `str` | Nom de l'optimiseur classique pour le QAOA. Tous les solveurs proposés par SciPy, listés [ici](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), sont utilisables. Tu devras configurer `classical_optimizer_options` en conséquence | Non | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Options de l'optimiseur classique. Pour une liste complète des options disponibles, consulte la [documentation SciPy](https://docs.scipy.org/doc/scipy/reference/) | Non | `{"maxiter": 60}` |
| `optimization_level` | `integer` | La profondeur du circuit QAOA | Non | `3` |
| `num_transpiler_runs` | `integer` | Nombre d'exécutions du transpileur | Non | `30` |
| `pass_manager_options` | `dictionary` | Options pour générer le gestionnaire de passes préconfiguré | Non | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Options de l'Estimator. Pour une liste complète des options disponibles, consulte la [documentation Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | Non | `None` |
| `sampler_options` | `dictionary` | Options du Sampler. Pour une liste complète des options disponibles, consulte la [documentation Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | Non | `None` |

- Les `estimator_options` par défaut sont :

|     Nom     |    Type     | Valeur |
|-------------|-------------|--------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Les `sampler_options` par défaut sont :

|     Nom     |    Type     | Valeur |
|-------------|-------------|--------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Utilisation

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### Validations
- `name` :
    - Le nom doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Aucun classificateur portant le même nom ne doit déjà exister dans le répertoire de données partagé.
### 3. Delete
L'action `delete` supprime un classificateur du répertoire de données partagé.

#### Entrées
|     Nom     |    Type     | Description | Requis |
|-------------|-------------|-------------|--------|
| `action`    | `str`    | Le nom de l'action. Doit être `delete`. | Oui |
| `name`      | `str`    | Le nom du classificateur à supprimer. | Oui |

#### Utilisation

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### Validations
- `name` :
    - Le nom doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Un classificateur portant le même nom doit déjà exister dans le répertoire de données partagé.
### 4. Fit
L'action `fit` entraîne un classificateur à l'aide des données d'entraînement fournies.

#### Entrées
|     Nom     |    Type     | Description | Requis |
|-------------|-------------|-------------|--------|
| `action`    | `str`    | Le nom de l'action. Doit être `fit`. | Oui |
| `name`      | `str`    | Le nom du classificateur à entraîner. | Oui |
| `X`         | `array` ou `list` ou `str` | Les données d'entraînement. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `y`         | `array` ou `list` ou `str` | Les valeurs cibles d'entraînement. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `fit_params`| `dictionary`| Paramètres supplémentaires à passer à la méthode `fit` du classificateur. | Non |

##### fit_params
|     Nom     |    Type     | Description | Requis | Défaut |
|-------------|-------------|-------------|--------|--------|
| `validation_data` | `tuple` | Les données et étiquettes de validation. | Non | `None` |
| `pos_label` | `integer` ou `str` | L'étiquette de classe à mapper sur 1. | Non | `None` |
| `optimization_data` | `str` | Jeu de données sur lequel optimiser l'ensemble. Peut être l'un des suivants : `train`, `validation`, `both`. | Non | `train` |

#### Utilisation

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### Validations
- `name` :
    - Le nom doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Un classificateur portant le même nom doit déjà exister dans le répertoire de données partagé.
### 5. Predict
L'action `predict` permet d'obtenir des prédictions dures (hard) et douces (soft) (probabilités).

#### Entrées
|     Nom     |    Type     | Description | Requis |
|-------------|-------------|-------------|--------|
| `action`    | `str`    | Le nom de l'action. Doit être `predict`. | Oui |
| `name`      | `str`    | Le nom du classificateur à utiliser. | Oui |
| `X`         | `array` ou `list` ou `str` | Les données de test. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `options["out"]` | `str` | Le nom du fichier JSON de sortie pour sauvegarder les prédictions dans le répertoire de données partagé. Si non fourni, les prédictions sont retournées dans le résultat de la tâche. | Non |

#### Utilisation

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### Validations
- `name` :
    - Le nom doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Un classificateur portant le même nom doit déjà exister dans le répertoire de données partagé.
- `options["out"]` :
    - Le nom du fichier doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Il doit avoir l'extension `.json`.
### 6. Fit-predict
L'action `fit_predict` entraîne un classificateur sur les données d'entraînement, puis l'utilise pour obtenir des prédictions dures et douces (probabilités).

#### Entrées
|     Nom     |    Type     | Description | Requis |
|-------------|-------------|-------------|--------|
| `action`    | `str`    | Le nom de l'action. Doit être `fit_predict`. | Oui |
| `name`      | `str`    | Le nom du classificateur à utiliser. | Oui |
| `X_train`   | `array` ou `list` ou `str` | Les données d'entraînement. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `y_train`   | `array` ou `list` ou `str` | Les valeurs cibles d'entraînement. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `X_test`    | `array` ou `list` ou `str` | Les données de test. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `fit_params`| `dictionary`| Paramètres supplémentaires à passer à la méthode `fit` du classificateur. | Non |
| `options["out"]` | `str` | Le nom du fichier JSON de sortie pour sauvegarder les prédictions dans le répertoire de données partagé. Si non fourni, les prédictions sont retournées dans le résultat de la tâche. | Non |

#### Utilisation

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### Validations
- `name` :
    - Le nom doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Un classificateur portant le même nom doit déjà exister dans le répertoire de données partagé.

- `options["out"]` :
    - Le nom du fichier doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Il doit avoir l'extension `.json`.
### 7. Create-fit-predict
L'action `create_fit_predict` crée un classificateur, l'entraîne sur les données d'entraînement fournies, puis l'utilise pour obtenir des prédictions dures et douces (probabilités).

#### Entrées
| Nom | Type | Description | Requis |
|-----|------|-------------|--------|
| `action` | `str` | Le nom de l'action parmi `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` et `delete`. | Oui |
| `name` | `str` | Le nom du classificateur à utiliser. | Oui |
| `quantum_classifier` | `str` | Le type du classificateur, c'est-à-dire `QuantumEnhancedEnsembleClassifier`. Par défaut `QuantumEnhancedEnsembleClassifier`. | Non |
| `X_train` | `array` ou `list` ou `str` | Les données d'entraînement. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `y_train` | `array` ou `list` ou `str` | Les valeurs cibles d'entraînement. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `X_test` | `array` ou `list` ou `str` | Les données de test. Cela peut être un tableau NumPy, une liste ou une chaîne référençant un nom de fichier dans le répertoire de données partagé. | Oui |
| `fit_params` | `dictionary` | Paramètres supplémentaires à passer à la méthode `fit` du classificateur. | Non |
| `options["save"]` | `boolean` | Indique s'il faut sauvegarder le classificateur entraîné dans le répertoire de données partagé. Par défaut `True`. | Non |
| `options["out"]` | `str` | Le nom du fichier JSON de sortie pour sauvegarder les prédictions dans le répertoire de données partagé. Si non fourni, les prédictions sont retournées dans le résultat de la tâche. | Non |

#### Utilisation

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### Validations
- `name` :
    - Si `options["save"]` est défini à `True` :
        - Le nom doit être unique, une chaîne d'au plus 64 caractères.
        - Il ne peut contenir que des caractères alphanumériques et des underscores.
        - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
        - Aucun classificateur portant le même nom ne doit déjà exister dans le répertoire de données partagé.

- `options["out"]` :
    - Le nom du fichier doit être unique, une chaîne d'au plus 64 caractères.
    - Il ne peut contenir que des caractères alphanumériques et des underscores.
    - Il doit commencer par une lettre et ne peut pas se terminer par un underscore.
    - Il doit avoir l'extension `.json`.
---
## Premiers pas
Authentifie-toi avec ta [clé API IBM Quantum Platform](http://quantum.cloud.ibm.com/) et sélectionne la Qiskit Function comme suit :

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## Exemple
Dans cet exemple, tu vas utiliser la fonction « Singularity Machine Learning - Classification » pour classifier un jeu de données composé de deux demi-cercles entrelacés en forme de lune. Le jeu de données est synthétique, bidimensionnel et étiqueté avec des étiquettes binaires. Il est conçu pour être difficile à traiter par des algorithmes tels que le clustering centroïde et la classification linéaire.
![Jeu de données moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
À travers ce processus, tu apprendras à créer le classificateur, à l'ajuster aux données d'entraînement, à l'utiliser pour prédire sur les données de test et à supprimer le classificateur une fois terminé.
Avant de commencer, tu dois installer [scikit-learn](https://scikit-learn.org/). Installe-le avec la commande suivante :

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


Effectue les étapes suivantes :

1) Crée le jeu de données synthétique en utilisant la fonction [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) de [scikit-learn](https://scikit-learn.org/).
2) Téléverse le jeu de données synthétique généré vers le [répertoire de données partagé](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).
3) Crée le classificateur amélioré quantiquement en utilisant l'action [create](#2-create).
4) Liste tes classificateurs en utilisant l'action [list](#1-list).
5) Entraîne le classificateur sur les données d'entraînement en utilisant l'action [fit](#4-fit).
6) Utilise le classificateur entraîné pour prédire sur les données de test en utilisant l'action [predict](#5-predict).
7) Supprime le classificateur en utilisant l'action [delete](#3-delete).
8) Nettoie une fois que tu as terminé.
**Étape 1.** Importe les modules nécessaires et génère le jeu de données synthétique, puis divise-le en ensembles d'entraînement et de test.

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**Étape 2.** Sauvegarde les jeux de données d'entraînement et de test étiquetés sur ton disque local, puis téléverse-les dans le [répertoire de données partagé](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**Étape 3.** Crée un classificateur amélioré quantiquement en utilisant l'action [create](#2-create).

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>